In [1]:
import pyspark
from pyspark.sql import SparkSession
import os
import sys
import optuna
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [ ]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName("GPU-Test")
    .master('spark://macbook:7077')
    .config("spark.executor.resource.gpu.amount", "2")
    .config("spark.task.resource.gpu.amount", "2")
    .config("spark.driver.resource.gpu.amount", "1")
    #.config("spark.executor.extraClassPath","${SPARK_RAPIDS_PLUGIN_JAR}")
    #.config("spark.driver.extraClassPath"  , "${SPARK_RAPIDS_PLUGIN_JAR}")
    #.config("spark.driver.resource.gpu.discoveryScript", "/opt/sparkRapidsPlugin/getGpusResources.sh")
    .config("spark.executor.resource.gpu.discoveryScript", "/opt/sparkRapidsPlugin/getGpusResources.sh")
   # .config("spark.plugins","com.nvidia.spark.SQLPlugin")
   # .config("spark.rapids.sql.explain","ALL")
    .getOrCreate()
)

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
        .appName("GPU-Test")
        .master("spark://macbook:7077")
        .config("spark.executor.resource.gpu.amount", "2")
        .config("spark.task.resource.gpu.amount", "2")
        .config("spark.executor.resource.gpu.discoveryScript", "/opt/sparkRapidsPlugin/getGpusResources.sh")
        # DO NOT set driver GPU configs on a MacBook
        .config("spark.driver.host", "macbook")
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.driver.port", "50000")
        .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 15:06:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

In [4]:
print(spark.sparkContext.getConf().get("spark.task.resource.gpu.amount"))

2


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

def train_mnist_ddp(learning_rate=0.01, epochs=5):
    # 1. Initialize the process group for distributed training
    import torch.distributed as dist
    import os
    
    dist.init_process_group("nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)

    # 2. Define Model
    class Net(nn.Module):
        def __init__(self):
            super(Net, self).__init__()
            self.conv1 = nn.Conv2d(1, 32, 3, 1)
            self.conv2 = nn.Conv2d(32, 64, 3, 1)
            self.fc1 = nn.Linear(9216, 128)
            self.fc2 = nn.Linear(128, 10)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            x = F.max_pool2d(F.relu(self.conv2(x)), 2)
            x = torch.flatten(x, 1)
            x = F.relu(self.fc1(x))
            return self.fc2(x)

    model = Net().cuda()
    model = DDP(model, device_ids=[local_rank])

    # 3. Load Data with DistributedSampler
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    
    train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
    sampler = DistributedSampler(train_dataset)
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=64, sampler=sampler
    )

    optimizer = optim.Adadelta(model.parameters(), lr=learning_rate)

    # 4. Training Loop
    model.train()
    for epoch in range(epochs):
        sampler.set_epoch(epoch)
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.cuda(), target.cuda()
            optimizer.zero_grad()
            output = model(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            optimizer.step()
            
            if batch_idx % 100 == 0 and local_rank == 0:
                print(f"Epoch {epoch} | Batch {batch_idx} | Loss: {loss.item():.4f}")

    # Clean up
    dist.destroy_process_group()

In [ ]:
from pyspark.ml.torch.distributor import TorchDistributor

# num_processes=3 matches your functional GPU count
distributor = TorchDistributor(num_processes=2, local_mode=False, use_gpu=True)

# Launch the training
distributor.run(train_mnist_ddp, learning_rate=0.001, epochs=10)

Started distributed training with 2 executor processes
